# ЛР-3. Ассоциативные правила

Датасет: Online Retail (UCI 352).  
Алгоритмы: Apriori и FP-Growth (реализация с нуля).  
Задачи: частые наборы, правила, рекомендации, сравнение времени.

In [ ]:
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from mlcore.data.datasets import load_lr3_dataset
from mlcore.utils.timing import timer

ROOT = Path.cwd()
for p in [ROOT, ROOT.parent, ROOT.parent.parent, ROOT.parent.parent.parent]:
    if (p / 'mlcore').exists():
        ROOT = p
        break
ASSETS = ROOT / 'labs/lr-3/assets'
ASSETS.mkdir(parents=True, exist_ok=True)

## 1. Загрузка и анализ данных

In [ ]:
ds = load_lr3_dataset()
df = ds.features.copy()
if ds.targets is not None:
    for col in ds.targets.columns:
        df[col] = ds.targets[col].values
print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head()

## 2. Предобработка транзакций

In [ ]:
# Определяем столбцы
invoice_col = [c for c in df.columns if 'invoice' in c.lower()][0]
desc_col = [c for c in df.columns if 'desc' in c.lower()][0]
qty_col = [c for c in df.columns if 'quant' in c.lower()][0]

print(f'Using: invoice={invoice_col}, desc={desc_col}, qty={qty_col}')

# Фильтрация
df_clean = df.copy()
df_clean[invoice_col] = df_clean[invoice_col].astype(str)
df_clean = df_clean[~df_clean[invoice_col].str.startswith('C')]  # cancelled
df_clean = df_clean.dropna(subset=[desc_col])
df_clean = df_clean[pd.to_numeric(df_clean[qty_col], errors='coerce').fillna(0) > 0]

# Формируем транзакции как frozenset
transactions = (
    df_clean.groupby(invoice_col)[desc_col]
    .apply(lambda items: frozenset(items.unique()))
    .tolist()
)

print(f'Transactions: {len(transactions)}')
print(f'Unique items: {len(set.union(*[set(t) for t in transactions]))}')
print(f'Avg basket size: {np.mean([len(t) for t in transactions]):.1f}')

## 3. Apriori

In [ ]:
def apriori(transactions: list[frozenset], min_support: float) -> dict[frozenset, int]:
    """Apriori algorithm. Returns {itemset: support_count}."""
    n = len(transactions)
    min_count = min_support * n

    # L1
    item_counts = Counter()
    for t in transactions:
        for item in t:
            item_counts[frozenset([item])] += 1
    Lk = {s: c for s, c in item_counts.items() if c >= min_count}
    all_frequent = dict(Lk)
    k = 2

    while Lk:
        # Generate candidates (join step)
        prev = list(Lk.keys())
        candidates = set()
        for i, a in enumerate(prev):
            for b in prev[i + 1:]:
                union = a | b
                if len(union) == k:
                    candidates.add(union)

        # Prune: all (k-1)-subsets must be frequent
        pruned = set()
        for c in candidates:
            if all(c - frozenset([item]) in Lk for item in c):
                pruned.add(c)

        # Count support
        counts = {c: 0 for c in pruned}
        for t in transactions:
            for c in pruned:
                if c.issubset(t):
                    counts[c] += 1

        Lk = {c: cnt for c, cnt in counts.items() if cnt >= min_count}
        all_frequent.update(Lk)
        k += 1

    return all_frequent

print('Apriori defined.')

In [ ]:
MIN_SUPPORT = 0.03

with timer('Apriori'):
    freq_apriori = apriori(transactions, MIN_SUPPORT)

# Stats per level
levels_a = Counter(len(s) for s in freq_apriori)
print(f'Total frequent itemsets (Apriori): {len(freq_apriori)}')
for k in sorted(levels_a):
    print(f'  Level {k}: {levels_a[k]} itemsets')

## 4. FP-Growth

In [ ]:
class FPNode:
    __slots__ = ('item', 'count', 'parent', 'children', 'next_link')
    def __init__(self, item, parent=None):
        self.item = item
        self.count = 0
        self.parent = parent
        self.children: dict = {}
        self.next_link = None


def _build_fp_tree(transactions, min_count):
    """Build FP-tree and header table."""
    freq = Counter()
    for t in transactions:
        freq.update(t)
    freq = {item: c for item, c in freq.items() if c >= min_count}
    if not freq:
        return None, {}

    root = FPNode(None)
    header: dict[str, FPNode] = {}  # item -> first node

    for t in transactions:
        ordered = sorted([item for item in t if item in freq], key=lambda x: (-freq[x], str(x)))
        node = root
        for item in ordered:
            if item not in node.children:
                child = FPNode(item, parent=node)
                node.children[item] = child
                # Update header linked list
                if item not in header:
                    header[item] = child
                else:
                    cur = header[item]
                    while cur.next_link is not None:
                        cur = cur.next_link
                    cur.next_link = child
            node = node.children[item]
            node.count += 1

    return root, header


def _fp_growth(header, prefix, min_count, frequent):
    """Recursive FP-Growth mining."""
    # Process items in ascending frequency
    items_sorted = sorted(header.keys(), key=lambda x: sum(
        n.count for n in _iter_nodes(header[x])))

    for item in items_sorted:
        new_prefix = prefix | frozenset([item])
        support = sum(n.count for n in _iter_nodes(header[item]))
        if support >= min_count:
            frequent[new_prefix] = support

            # Conditional pattern base
            cond_patterns = []
            for node in _iter_nodes(header[item]):
                path = []
                parent = node.parent
                while parent is not None and parent.item is not None:
                    path.append(parent.item)
                    parent = parent.parent
                for _ in range(node.count):
                    cond_patterns.append(path)

            # Conditional FP-tree
            _, cond_header = _build_fp_tree(cond_patterns, min_count)
            if cond_header:
                _fp_growth(cond_header, new_prefix, min_count, frequent)


def _iter_nodes(start):
    node = start
    while node is not None:
        yield node
        node = node.next_link


def fp_growth(transactions: list[frozenset], min_support: float) -> dict[frozenset, int]:
    """FP-Growth algorithm. Returns {itemset: support_count}."""
    n = len(transactions)
    min_count = min_support * n
    _, header = _build_fp_tree(transactions, min_count)
    frequent: dict[frozenset, int] = {}
    if header:
        _fp_growth(header, frozenset(), min_count, frequent)
    return frequent

print('FP-Growth defined.')

In [ ]:
with timer('FP-Growth'):
    freq_fpg = fp_growth(transactions, MIN_SUPPORT)

levels_f = Counter(len(s) for s in freq_fpg)
print(f'Total frequent itemsets (FP-Growth): {len(freq_fpg)}')
for k in sorted(levels_f):
    print(f'  Level {k}: {levels_f[k]} itemsets')

# Verification
assert set(freq_apriori.keys()) == set(freq_fpg.keys()), 'Itemset mismatch!'
print('\nVerification: Apriori and FP-Growth produce identical itemsets.')

## 5. Генерация правил

In [ ]:
def generate_rules(frequent: dict[frozenset, int], n_transactions: int,
                   min_confidence: float = 0.5) -> list[dict]:
    """Generate association rules from frequent itemsets."""
    rules = []
    for itemset, count in frequent.items():
        if len(itemset) < 2:
            continue
        support = count / n_transactions
        for item in itemset:
            antecedent = itemset - frozenset([item])
            consequent = frozenset([item])
            if antecedent in frequent:
                confidence = count / frequent[antecedent]
                if confidence >= min_confidence and consequent in frequent:
                    lift = confidence / (frequent[consequent] / n_transactions)
                    rules.append({
                        'antecedent': antecedent,
                        'consequent': consequent,
                        'support': round(support, 4),
                        'confidence': round(confidence, 4),
                        'lift': round(lift, 2),
                    })
    return sorted(rules, key=lambda r: r['lift'], reverse=True)

rules = generate_rules(freq_apriori, len(transactions), min_confidence=0.5)
print(f'Rules generated: {len(rules)}')

# Top 20
rules_df = pd.DataFrame(rules[:20])
rules_df['antecedent'] = rules_df['antecedent'].apply(lambda s: ', '.join(sorted(s)))
rules_df['consequent'] = rules_df['consequent'].apply(lambda s: ', '.join(sorted(s)))
rules_df

## 6. Сравнение производительности

In [ ]:
import time

t0 = time.perf_counter()
_ = apriori(transactions, MIN_SUPPORT)
time_apriori = time.perf_counter() - t0

t0 = time.perf_counter()
_ = fp_growth(transactions, MIN_SUPPORT)
time_fpg = time.perf_counter() - t0

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(['Apriori', 'FP-Growth'], [time_apriori, time_fpg], color=['#2c7fb8', '#7fc97f'])
ax.set_ylabel('Time (s)')
ax.set_title(f'Training Time Comparison (min_support={MIN_SUPPORT})')
for i, t in enumerate([time_apriori, time_fpg]):
    ax.text(i, t + 0.01, f'{t:.2f}s', ha='center')
fig.tight_layout()
fig.savefig(ASSETS / 'timing_comparison.png', dpi=150)
plt.close(fig)
print(f'Apriori: {time_apriori:.2f}s, FP-Growth: {time_fpg:.2f}s')
print('Saved timing_comparison.png')

## 7. Рекомендательная функция

In [ ]:
def recommend(basket: set[str], rules: list[dict], top_n: int = 5) -> list[tuple[str, float]]:
    """Recommend items based on basket contents and association rules."""
    candidates = {}
    for rule in rules:
        if rule['antecedent'].issubset(basket):
            for item in rule['consequent']:
                if item not in basket:
                    if item not in candidates or candidates[item] < rule['confidence']:
                        candidates[item] = rule['confidence']
    return sorted(candidates.items(), key=lambda x: x[1], reverse=True)[:top_n]

# Примеры
if rules:
    sample_item = list(list(rules[0]['antecedent'])[:1])
    print(f'Basket: {sample_item}')
    recs = recommend(set(sample_item), rules)
    for item, conf in recs:
        print(f'  -> {item} (confidence={conf:.2f})')

## 8. Выводы

1. Apriori и FP-Growth реализованы с нуля, дают идентичные результаты.
2. FP-Growth, как правило, быстрее за счёт отсутствия генерации кандидатов.
3. Рекомендательная функция находит релевантные товары по правилам с высоким lift.
4. min_support выбран для баланса информативности и вычислительной сложности.